# Top-K 采样

> 本文件原为空，按"详细解析 + 骨架"补全。

## 1. 原理（Fan et al. 2020）
1. 取 logits 最大的 K 个，其余置 $-\infty$；
2. softmax 得到只在 K 个上的概率；
3. 按该分布 multinomial 采样。

## 2. 直觉
- K=1 退化为贪心；K=词表大小 退化为纯采样。
- 在"质量"与"多样性"间折中：只在高概率候选里随机，避免低概率垃圾 token。
- K 常取 40~50；过大接近纯采样（噪声多），过小退化贪心。

## 3. 考察点
- `torch.topk` 取值与索引、`scatter_` 置 -inf
- 与温度的组合顺序（先除温度再 TopK）
- 数值稳定（softmax 内部已 safe）

In [ ]:
import torch
import torch.nn.functional as F

def topk_sampling(logits, k=50, temperature=1.0):
    """
    logits: [batch, vocab] 或 [vocab]
    返回采样到的 token id。
    """
    # TODO:
    # logits = logits / temperature
    # topk_logits, topk_idx = torch.topk(logits, k, dim=-1)
    # probs = F.softmax(topk_logits, dim=-1)
    # sampled = torch.multinomial(probs, num_samples=1)
    # next_token = topk_idx.gather(-1, sampled)
    raise NotImplementedError

# 验证：k=1 时应等价 argmax；多次调用结果有随机性

## 小结
- TopK 的候选集合大小固定 K，对"长尾分布"与"尖锐分布"用同一 K 不够自适应——这正是 TopP 要解决的。
- 实现关键：在**子集**上 softmax 再 gather 回原词表索引，不要对全词表 softmax 再 mask（数值上等价但低效）。